### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part6_autodiff_tests.py

import mugrade
import os
from part6_autodiff_tests import *
os.environ["MUGRADE_HW"] = "Part 6 - Automatic Differentiation"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

### Download the necessary files
from huggingface_hub import hf_hub_download

filenames = [
    'config.d12.json',
    'llm.d12.pt',
    'fineweb-edu-10BT.shuffle.bin'
]
for f in filenames:
    if not os.path.exists(f):
        hf_hub_download(repo_id="zkolter/llm_speedrun", filename=f, repo_type="dataset", local_dir=".")

### Original LLM Architecture

In this first portion of the notebook, you'll copy over your existing implementation and save the gradients.  Specifically, copy the code for the LLM class (and support functions) as well as the `cross_entropy_loss` function below.

In [ ]:
### BEGIN YOUR CODE
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename is not None:
            with open(filename, "rt") as fobj:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(fobj)
                self.merges = {
                    tuple(k): v for k, v in zip(merge_keys, merge_values)
                }
                self.special_tokens = {
                    int(k): v for k, v in self.special_tokens.items()
                }
        else:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i+1] == b:
                word[i: i+2] = [merged]
            i+=1
        ### END YOUR CODE

    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k, v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        regex_matching = re.findall(r"\s\S+|\s+|\S+", text)
        frequency_counter = Counter(regex_matching)
        frequency_map = {
            self.replace_special_tokens(k): v for k, v in frequency_counter.items() if v > 1
        }

        # Build word_list_frequency_map without defaultdict
        word_list_frequency_map = {}

        for word, freq in frequency_map.items():
            filtered_word = tuple(
                ord(char) for char in word
                if ord(char) not in self.special_tokens
            )

            if filtered_word:
                word_list_frequency_map[filtered_word] = \
                    word_list_frequency_map.get(filtered_word, 0) + freq

        words_as_lists, words_as_lists_frequencies = \
            [list(i) for i in word_list_frequency_map.keys()], list(word_list_frequency_map.values())


        for _ in range(len(self.vocab), target_vocab_size):
            existing_vocab_size = len(self.vocab)
            suitable_merge_pairs = Counter()

            for current_word_as_list, current_word_frequency in \
                zip(words_as_lists, words_as_lists_frequencies):

                for i in range(len(current_word_as_list) - 1):
                    suitable_merge_pairs[(current_word_as_list[i], current_word_as_list[i+1])] +=\
                        current_word_frequency

            # Break condition for testing
            # if len(suitable_merge_pairs) == 0:
            #     break

            symbol_a, symbol_b = suitable_merge_pairs.most_common()[0][0]
            self.vocab.append(self.vocab[symbol_a] + self.vocab[symbol_b])
            self.merges[(symbol_a, symbol_b)] = existing_vocab_size

            for word_as_list in words_as_lists:
                self.merge_pair(word_as_list, symbol_a, symbol_b, existing_vocab_size)

        if len(self.special_tokens) != 0:
            max_special_token = max(self.special_tokens.keys())
            self.vocab += [""] * (max_special_token - len(self.vocab) + 1)
            for k, v in self.special_tokens.items():
                self.vocab[k] = v
        ### END YOUR CODE

    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []

        for word in re.findall(r"\s\S+|\s+|\S+", text):
            word_as_list = [ord(char) for char in self.replace_special_tokens(word)]
            # Special tokens are now single code points (e.g. chr(32760)). No merge
            # contains them, so they pass through as their own token IDs.

            if  len(word_as_list) == 0:
                continue

            while True:
                word_symbol_pairs_to_vocab_index_mapping = {}
                for i in range(len(word_as_list) - 1):
                    current_symbol_pair = \
                        (word_as_list[i], word_as_list[i+1])
                    if current_symbol_pair in self.merges:
                        word_symbol_pairs_to_vocab_index_mapping[current_symbol_pair] = \
                            self.merges[current_symbol_pair]

                if len(word_symbol_pairs_to_vocab_index_mapping) == 0:
                    break

                symbol_a, symbol_b = min_symbol_pair = min(
                    word_symbol_pairs_to_vocab_index_mapping,
                    key = word_symbol_pairs_to_vocab_index_mapping.get
                )
                self.merge_pair(word_as_list, symbol_a, symbol_b,
                                word_symbol_pairs_to_vocab_index_mapping[min_symbol_pair])

            tokens += word_as_list

        return tokens
        ### END YOUR CODE

    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[token] for token in tokens if token < len(self.vocab)])
        ### END YOUR CODE

    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as fobj:
            json.dump([
                self.vocab,
                list(self.merges.keys()),
                list(self.merges.values()),
                self.special_tokens
            ], fobj)
        ### END YOUR CODE
### END YOUR CODE


import torch
import torch.nn.functional as F

import math

def embedding(tokens, weight, dtype):
    ### BEGIN YOUR CODE
    return weight[tokens].to(dtype=dtype)
    ### END YOUR CODE

def linear(x, weight):
    ### BEGIN YOUR CODE
    return x @ weight.type_as(x)
    ### END YOUR CODE

def silu(x):
    ### BEGIN YOUR CODE
    return x / (1 + torch.exp(-x))
    ### END YOUR CODE

def rms_norm(x):
    ### BEGIN YOUR CODE
    x_typed = x.float()
    x_normed = x_typed / (
        (x_typed**2)
            .mean(
                dim = -1,
                keepdim = True,
            )
            .sqrt()
    )
    return x_normed.type_as(x)
    ### END YOUR CODE

def softmax(x):
    ### BEGIN YOUR CODE
    x_numerical_stable = \
        x - x.max(dim = -1, keepdim=True).values
    x_exp = x_numerical_stable.exp()
    return x_exp / (
        x_exp.sum(dim=-1, keepdim=True)
    )
    ### END YOUR CODE

def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    attention_score = softmax(
        q @ k.transpose(-2, -1) / math.sqrt(k.size(-1)) + mask
    )
    return attention_score @ v
    ### END YOUR CODE


class LLM:
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        L = x.shape[-2]
        w1 = self.buffers["rope_w1"][pos:pos + L]
        w2 = self.buffers["rope_w2"][pos:pos + L]
        flipped = x.reshape(-1, 2).flip(-1).reshape_as(x)
        return (w1 * x + w2 * flipped).type_as(x)
        ### END YOUR CODE

    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L, D = x.shape
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])
        wp = self.params[f"wp_{layer}"]

        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        q = rms_norm(self.rope(q, pos))
        k = rms_norm(self.rope(k, pos))

        if cache is not None:
            c = cache[layer]
            c["k"][:, :, pos:pos + L] = k
            c["v"][:, :, pos:pos + L] = v
            k = c["k"][:, :, :pos + L]
            v = c["v"][:, :, :pos + L]

        attn_output = self_attn(q, k, v, mask)

        return linear(
            attn_output.transpose(1, 2).reshape(B, L, D),
            wp
        )
        ### END YOUR CODE

    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        return linear(
            silu(
                linear(
                    x,
                    self.params[f"w1_{layer}"]
                )
            ),
            self.params[f"w2_{layer}"]
        )
        ### END YOUR CODE

    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        x = x + \
            self.multihead_attn(
                rms_norm(x), layer, mask, pos=pos, cache=cache
            )
        return x + self.mlp(
            rms_norm(x), layer
        )
        ### END YOUR CODE

    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        self.dtype = getattr(torch, config["dtype"])

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]

        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]

        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        self.params = {}
        dim_scale, dim_scale_mlp = \
            math.sqrt(2/hidden_dim), math.sqrt(2/self.mlp_dim)

        self.params["embedding"] = torch.randn(
            vocab_size, hidden_dim
        ) * dim_scale
        self.params["output"] = torch.randn(
            hidden_dim, vocab_size
        ) * dim_scale

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wk_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wv_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wp_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"w1_{i}"] = \
                torch.randn(
                    hidden_dim, self.mlp_dim
                ) * dim_scale
            self.params[f"w2_{i}"] = \
                torch.randn(
                    self.mlp_dim, hidden_dim
                ) * dim_scale_mlp

        self.buffers = {}
        mask = torch.full(
            (sequence_len, sequence_len),
            -float("inf"))
        self.buffers["mask"] = \
            torch.triu(mask, diagonal=1)


        freqs = config["rope_theta"] ** (
            (-2 * torch.arange(
                self.head_dim // 2
            ) / self.head_dim)
        )
        theta = torch.outer(
            torch.arange(sequence_len),
            freqs
        )

        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope_w1"] = \
            torch.stack([cos, cos], dim = -1).reshape(
                sequence_len, self.head_dim
            )
        self.buffers["rope_w2"] = \
            torch.stack([-sin, sin], dim = -1).reshape(
                sequence_len, self.head_dim
            )

        ### END YOUR CODE

    def __call__(self, tokens, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L = tokens.shape
        x = rms_norm(
            embedding(
                tokens,
                self.params["embedding"],
                self.dtype
            )
        )

        mask = self.buffers["mask"][pos:pos + L, :pos + L]

        for i in range(self.num_layers):
            x = self.transformer_block(
                x, i, mask, pos=pos, cache=cache
            )

        x_final = \
            rms_norm(x)

        return linear(
            x_final,
            self.params["output"]
        )
        ### END YOUR CODE

    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = \
            torch.load(filename, weights_only=True, map_location="cpu")
        ### END YOUR CODE

    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None, tool_eval=None, bpe=None):
        batch_size = prompts.shape[0]
        prompt_len = prompts.shape[1]

        start_pos, current_pos = 0, prompt_len

        if bpe is not None:
            mask_token_id = bpe.vocab.index("<MASK/>")
            tool_start_id = bpe.vocab.index("<TOOL>")
            tool_end_id = bpe.vocab.index("</TOOL>")
        else:
            mask_token_id = 0

        tokens = torch.cat([
            prompts,
            torch.full((batch_size, max_tokens), mask_token_id, device=prompts.device, dtype=prompts.dtype)
        ], dim=-1)

        max_seq_len = tokens.shape[1]

        kv_cache = [{
            "k": torch.zeros(batch_size, self.num_heads, max_seq_len, self.head_dim, device=prompts.device, dtype=self.dtype),
            "v": torch.zeros(batch_size, self.num_heads, max_seq_len, self.head_dim, device=prompts.device, dtype=self.dtype)
        } for _ in range(self.num_layers)]

        is_finished = [False] * batch_size

        with torch.no_grad():
            while current_pos < max_seq_len and not all(is_finished):
                logits = self(tokens[:, start_pos:current_pos], pos=start_pos, cache=kv_cache)[:, -1]
                probs = F.softmax(logits.float() / temp, dim=-1)

                start_pos = current_pos
                current_pos = current_pos + 1

                for batch_idx in range(batch_size):
                    if tokens[batch_idx, start_pos] == mask_token_id and not is_finished[batch_idx]:
                        sampled_token = torch.multinomial(probs[batch_idx], num_samples=1)
                        tokens[batch_idx, start_pos] = sampled_token

                        if eos is not None:
                            is_finished[batch_idx] = (tokens[batch_idx, start_pos] == eos).item()

                        if tool_eval is not None and tokens[batch_idx, start_pos] == tool_end_id:
                            tool_locs = torch.where(tokens[batch_idx, :start_pos] == tool_start_id)[0]

                            if len(tool_locs) > 0:
                                last_tool_start = tool_locs[-1]
                                tool_call_tokens = tokens[batch_idx, last_tool_start + 1 : start_pos].tolist()

                                response_str = tool_eval(bpe.decode(tool_call_tokens))
                                response_tokens = bpe.encode(f"<RESPONSE>{response_str}</RESPONSE>")

                                response_len = len(response_tokens)

                                if start_pos + response_len + 1 < max_seq_len:
                                    tokens[batch_idx, start_pos + 1 : start_pos + 1 + response_len] = torch.tensor(
                                        response_tokens, device=tokens.device
                                    )

        return tokens

import random
import wandb
import time
from array import array
import os

def cross_entropy_loss(logits, y, mask_token=-1, weights=1):
    ### BEGIN YOUR CODE
    logits_normed = logits - logits.max(dim = -1, keepdim = True)[0]
    loss = -logits_normed.take_along_dim(y.unsqueeze(-1), dim = -1)
    loss += logits_normed.exp().sum(dim = -1, keepdim = True).log()
    return (
        (weights * loss)[y != mask_token].sum() / \
        (y != mask_token).float().sum()
    )

    ### END YOUR CODE

class Adam:
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        self.lr = lr
        self.betas = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.params = params

        self.t = 1

        self.u = {k: torch.zeros_like(v) for k,v in self.params.items()}
        self.v = {k: tensor.clone().detach() for k, tensor in self.u.items()}

        for param in self.params.values():
            param.requires_grad_()
        ### END YOUR CODE

    def step(self):
        ### BEGIN YOUR CODE
        with torch.no_grad():
            for k,p in self.params.items():
                self.u[k] = self.betas[0]*self.u[k] + (1-self.betas[0])*p.grad
                self.v[k] = self.betas[1]*self.v[k] + (1-self.betas[1])*p.grad**2

                u_hat = self.u[k] / (1 - self.betas[0]**self.t)
                v_hat = self.v[k] / (1 - self.betas[1]**self.t)

                p *= (1 - self.lr * self.weight_decay)
                p -= self.lr * u_hat / (v_hat.sqrt() + self.eps)

                p.grad.zero_()
            self.t += 1
        ### END YOUR CODE

def finetune_llm_sft(rank, nccl_uid, config):
    ### BEGIN YOUR CODE
    torch.cuda.set_device(rank)

    with open(config["llm_config"]) as fobj:
        llm_config = json.load(fobj)

    llm = LLM(llm_config)
    llm.load(config["infilename"])

    for k in llm.params:
        llm.params[k] = llm.params[k].detach().cuda()
    for k in llm.buffers:
        llm.buffers[k] = llm.buffers[k].cuda()

    comm = torch.cuda.nccl.init_rank(config["num_gpus"], nccl_uid, rank)
    opt = Adam(llm.params, lr=config["lr"])

    with open(config["datafile"]) as f:
        dataset = json.load(f)

    bpe = BPE(config["tokenizer"])
    mask_token = bpe.vocab.index("<MASK/>")

    for _ in range(config["epochs"]):
        for i in range(0, len(dataset), config["batch_size"]):
            local_batch_size = (min(config["batch_size"], len(dataset) - i) + config["num_gpus"] - 1) // config["num_gpus"]
            data = dataset[i + rank * local_batch_size : i + (rank + 1) * local_batch_size]

            # sft loss
            tokens = [t["tokens"] for t in data]
            for j in range(len(tokens)):
                shift = random.randint(0, config["seq_len"] - tokens[j].index(mask_token) - 1)
                tokens[j] = tokens[j][-shift:] + tokens[j][:-shift]
            tokens = torch.tensor(tokens).cuda()
            logits = llm(tokens[:, :-1]).float()
            loss = cross_entropy_loss(logits, tokens[:, 1:], mask_token=mask_token)
            loss.backward()

            for p in llm.params.values():
                torch.cuda.nccl.all_reduce([p.grad], comms=[comm])
                p.grad /= config["num_gpus"]
            opt.step()

            torch.cuda.nccl.all_reduce([loss], comms=[comm])

    if rank == 0:
        llm.save(config["filename"])
    ### END YOUR CODE

After you have done this, the following code will save the gradient of the cross entropy loss with respect to the model parameters for the first 4 sequences in the fineweb EDU dataset.

In [ ]:
## Compute reference gradients on the first 4 sequences in fineweb
from array import array

with open("fineweb-edu-10BT.shuffle.bin", "rb") as f:
    tokens = array("H", f.read(4*(2048+1)*2)).tolist()
tokens = torch.tensor(tokens).reshape(4,2049).cuda()

with open("config.d12.json", "rt") as f: config = json.load(f)
llm = LLM(config)
llm.load("llm.d12.pt")
for k,p in llm.params.items():
    llm.params[k] = p.detach().cuda().requires_grad_()
for k,p in llm.buffers.items():
    llm.buffers[k] = llm.buffers[k].cuda()

logits = llm(tokens[:,:-1]).float()
loss = cross_entropy_loss(logits, tokens[:,1:])
loss.backward()

grads = {k:p.grad for k,p in llm.params.items()}
torch.save(grads, "grads.pt")

### Automatic differentiation

Now, implement the automatic differentiation library presented in class.  The additional functions you will need to implement, besides those presented in the class, are the `silu()`, `cross_entropy_loss` and `rms_norm` functions.  You'll have to work out the implementation of these functions.

Local tests use small NumPy arrays and need no dataset or saved model. Tensor operation tests check the forward and vector-Jacobian-product callbacks registered with `Tensor.build`. Implement the Tensor engine before testing the adapted LLM. Uncomment `# @mugrade.local_tests` to test a function, or change it to `@mugrade.submit_tests` to submit it. Keep the `_mugrade_name` decorators: they give the two constructors distinct grading names. The copied PyTorch LLM above is not graded in this assignment.


In [70]:
# Remove the PyTorch name so that no code below can use it by accident.
import torch
del torch

# CuPy mirrors the NumPy API on the GPU, so it is imported under the name `np`.
import cupy as np
import math

class Tensor:
    @mugrade.submit_tests
    @_mugrade_name("Tensor_init")
    def __init__(self, value, vjp=None, parents=None, requires_grad=False):
        ### BEGIN YOUR CODE
        self.value = value
        self.vjp = vjp
        self.parents = parents
        self.requires_grad = requires_grad

        self.grad = None
        self.num_children = 0
        ### END YOUR CODE

    @staticmethod
    @mugrade.submit_tests
    def build(forward, vjp, *args):
        ### BEGIN YOUR CODE
        values = [
            arg.value if isinstance(arg, Tensor) else arg
            for arg in args
        ]

        value = forward(*values)

        has_tensor = any(isinstance(arg, Tensor) for arg in args)
        requires_grad = any(
            isinstance(arg, Tensor) and arg.requires_grad
            for arg in args
        )

        if not has_tensor:
            return value

        if requires_grad:
            for arg in args:
                if isinstance(arg, Tensor):
                    arg.num_children += 1

            return Tensor(
                value,
                vjp=vjp,
                parents=args,
                requires_grad=True,
            )

        return Tensor(value)
        ### END YOUR CODE

    @mugrade.submit_tests
    def __add__(self, other):
        ### BEGIN YOUR CODE
        def forward(x, y):
            return x + y
        def vjp(grad, out, x, y):
            return grad, grad
        return Tensor.build(
            forward,
            vjp,
            self,
            other
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def __mul__(self, other):
        ### BEGIN YOUR CODE
        def forward(x, y):
            return x * y
        def vjp(grad, out, x, y):
            return grad * y, grad * x
        return Tensor.build(
            forward,
            vjp,
            self,
            other
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def __matmul__(self, other):
        ### BEGIN YOUR CODE
        def forward(x, y):
            return x @ y

        def vjp(grad, out, x, y):
            dx = grad @ y.swapaxes(-1, -2)
            dy = x.swapaxes(-1, -2) @ grad

            while dx.ndim > x.ndim:
                dx = dx.sum(axis=0)
            while dy.ndim > y.ndim:
                dy = dy.sum(axis=0)

            return dx, dy

        return Tensor.build(
            forward,
            vjp,
            self,
            other
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def __getitem__(self, idx):
        ### BEGIN YOUR CODE
        def forward(x):
            return x[idx]

        def vjp(grad, out, x):
            g = np.zeros_like(x)
            if isinstance(idx, np.ndarray):
                np.add.at(g, idx, grad)
            else:
                g[idx] = grad
            return g,

        return Tensor.build(forward, vjp, self)

        ### END YOUR CODE

    @mugrade.submit_tests
    def __setitem__(self, idx, x):
        ### BEGIN YOUR CODE
        self.value[idx] = x.value
        ### END YOUR CODE

    @mugrade.submit_tests
    def transpose(self, i, j):
        ### BEGIN YOUR CODE
        def forward(x):
            return x.swapaxes(i, j)

        def vjp(grad, out, x):
            return grad.swapaxes(i, j),

        return Tensor.build(forward, vjp, self)
        ### END YOUR CODE

    @mugrade.submit_tests
    def reshape(self, *shape):
        ### BEGIN YOUR CODE
        def forward(x):
            return x.reshape(*shape)

        def vjp(grad, out, x):
            return grad.reshape(x.shape),

        return Tensor.build(forward, vjp, self)
        ### END YOUR CODE

    @mugrade.submit_tests
    def flip(self, dim):
        ### BEGIN YOUR CODE
        def forward(x):
            return np.flip(x, dim)

        def vjp(grad, out, x):
            return np.flip(grad, dim),

        return Tensor.build(forward, vjp, self)
        ### END YOUR CODE

    @mugrade.submit_tests
    def to(self, dtype):
        ### BEGIN YOUR CODE
        def forward(x):
            return x.astype(dtype)

        def vjp(grad, out, x):
            return grad.astype(x.dtype),

        return Tensor.build(forward, vjp, self)
        ### END YOUR CODE


    @property
    @mugrade.submit_tests
    def shape(self):
        ### BEGIN YOUR CODE
        return self.value.shape
        ### END YOUR CODE

    @property
    @mugrade.submit_tests
    def dtype(self):
        ### BEGIN YOUR CODE
        return self.value.dtype
        ### END YOUR CODE

    @mugrade.submit_tests
    def backward(self):
        ### BEGIN YOUR CODE
        if self.grad is None:
            self.grad = 1

        if self.parents:
            # Tensor.build accepts plain arrays as arguments (for example,
            # integer targets). Pass them through unchanged; they get no grad.
            args = [
                getattr(a, "value", a) for a in self.parents
            ]

            grads = self.vjp(
                self.grad, self.value, *args
            )

            for p, g in zip(self.parents, grads):
                if getattr(p, "requires_grad", False):
                    if p.grad is None:
                        p.grad = g
                    else:
                        p.grad = p.grad + g

                    p.num_children -= 1

                    if p.num_children == 0:
                        p.backward()
        ### END YOUR CODE


@mugrade.submit_tests
def softmax(x):
    ### BEGIN YOUR CODE
    def forward(x):
        z = np.exp(x - x.max(axis=-1, keepdims=True))
        return z / z.sum(axis=-1, keepdims=True)

    def vjp(grad, out, x):
        dx = out * (grad - (out * grad).sum(axis=-1, keepdims=True))
        return dx,

    return Tensor.build(forward, vjp, x)
    ### END YOUR CODE


@mugrade.submit_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    def forward(x, targets):
        targets = targets.astype(np.int64).reshape(x.shape[:-1] + (1,))
        z = x - x.max(axis=-1, keepdims=True)
        log_probs = z - np.log(np.exp(z).sum(axis=-1, keepdims=True))
        return -np.take_along_axis(log_probs, targets, axis=-1).mean()

    def vjp(grad, out, x, targets):
        targets = targets.astype(np.int64).reshape(x.shape[:-1] + (1,))
        z = np.exp(x - x.max(axis=-1, keepdims=True))
        probs = z / z.sum(axis=-1, keepdims=True)
        np.put_along_axis(
            probs, targets,
            np.take_along_axis(probs, targets, axis=-1) - 1,
            axis=-1,
        )
        return grad * probs / targets.size,

    return Tensor.build(forward, vjp, logits, y)
    ### END YOUR CODE


@mugrade.submit_tests
def silu(x):
    ### BEGIN YOUR CODE
    def forward(x):
        sigmoid = 1 / (1 + np.exp(-x))
        return x * sigmoid

    def vjp(grad, out, x):
        sigmoid = 1 / (1 + np.exp(-x))
        return grad * sigmoid * (1 + x * (1 - sigmoid)),

    return Tensor.build(forward, vjp, x)
    ### END YOUR CODE


@mugrade.submit_tests
def rms_norm(x):
    ### BEGIN YOUR CODE
    def forward(x):
        x32 = x.astype(np.float32)
        rms = np.sqrt(np.mean(x32 * x32, axis=-1, keepdims=True))
        return (x32 / rms).astype(x.dtype)

    def vjp(grad, out, x):
        x32 = x.astype(np.float32)
        g32 = grad.astype(np.float32)
        mean_sq = np.mean(x32 * x32, axis=-1, keepdims=True)
        dx = (
            g32 - x32 * np.mean(g32 * x32, axis=-1, keepdims=True) / mean_sq
        ) / np.sqrt(mean_sq)
        return dx.astype(x.dtype),

    return Tensor.build(forward, vjp, x)
    ### END YOUR CODE


@mugrade.submit_tests
def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    def forward(q, k, v, mask):
        scale = math.sqrt(q.shape[-1])
        scores = q @ k.swapaxes(-1, -2) / scale + mask
        return softmax(scores) @ v

    def vjp(grad, out, q, k, v, mask):
        scale = math.sqrt(q.shape[-1])
        scores = q @ k.swapaxes(-1, -2) / scale + mask
        a = softmax(scores)

        dv = a.swapaxes(-1, -2) @ grad
        da = grad @ v.swapaxes(-1, -2)
        dscores = a * (
            da - (a * da).sum(axis=-1, keepdims=True)
        )

        dq = dscores @ k / scale
        dk = dscores.swapaxes(-1, -2) @ q / scale
        return dq, dk, dv

    return Tensor.build(forward, vjp, q, k, v, mask)
    ### END YOUR CODE

Mugrade: Submitting tests for function Tensor_init():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Mugrade: Submitting tests for function build():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Mugrade: Submitting tests for function __add__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function __mul__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function __matmul__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function __getitem__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function __setitem__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function transpose():
Grader test 1 passed
Grader test 2 passed
Grader test 3 pass

Now, adapt the LLM implementation above to use your automatic differentiation engine.  You will want to ensure that none of the code uses torch, even to build the buffers.  Most of this should be exactly the same as your existing LLM implementation, and here we test only the functions that are different.  Note that you don't need to implement the `generate` or load/save functionality in this version of the LLM, as it won't be used directly.

In [79]:
@mugrade.submit_tests
def embedding(x, weights, dtype):
    ### BEGIN YOUR CODE
    return weights[x].to(dtype)
    ### END YOUR CODE

@mugrade.submit_tests
def linear(x, weights):
    ### BEGIN YOUR CODE
    return x @ weights.to(x.dtype)
    ### END YOUR CODE

class LLM:
    @mugrade.submit_tests
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        L = x.shape[-2]
        w1 = self.buffers["rope1"][pos:pos + L]
        w2 = self.buffers["rope2"][pos:pos + L]

        flipped = x.reshape(-1, 2).flip(-1).reshape(*x.shape)
        return (x * w1 + flipped * w2).to(x.dtype)
        ### END YOUR CODE

    @mugrade.submit_tests
    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L, D = x.shape

        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])

        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        q = rms_norm(self.rope(q, pos=pos))
        k = rms_norm(self.rope(k, pos=pos))

        if cache is not None:
            c = cache[layer]
            c["k"][:, :, pos:pos + L] = k.value
            c["v"][:, :, pos:pos + L] = v.value
            k = c["k"][:, :, :pos + L]
            v = c["v"][:, :, :pos + L]

        out = self_attn(q, k, v, mask)
        out = out.transpose(1, 2).reshape(B, L, D)
        return linear(out, self.params[f"wp_{layer}"])
        ### END YOUR CODE

    @mugrade.submit_tests
    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        hidden = silu(linear(x, self.params[f"w1_{layer}"]))
        return linear(hidden, self.params[f"w2_{layer}"])
        ### END YOUR CODE

    @mugrade.submit_tests
    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        x = x + self.multihead_attn(
            rms_norm(x), layer, mask, pos=pos, cache=cache
        )
        return x + self.mlp(rms_norm(x), layer)
        ### END YOUR CODE

    @mugrade.submit_tests
    @_mugrade_name("LLM_init")
    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        self.dtype = config["dtype"]

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]
        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]
        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        dim_scale = math.sqrt(2 / hidden_dim)
        dim_scale_mlp = math.sqrt(2 / self.mlp_dim)

        def parameter(shape, scale):
            value = (np.random.randn(*shape) * scale).astype(np.float32)
            return Tensor(value, requires_grad=True)

        self.params = {
            "embedding": parameter((vocab_size, hidden_dim), dim_scale),
            "output": parameter((hidden_dim, vocab_size), dim_scale),
        }

        for i in range(self.num_layers):
            for name in ("wq", "wk", "wv", "wp"):
                self.params[f"{name}_{i}"] = parameter(
                    (hidden_dim, hidden_dim), dim_scale
                )
            self.params[f"w1_{i}"] = parameter(
                (hidden_dim, self.mlp_dim), dim_scale
            )
            self.params[f"w2_{i}"] = parameter(
                (self.mlp_dim, hidden_dim), dim_scale_mlp
            )

        self.buffers = {}

        # Build the buffers in float64, then cast them to the model dtype.
        upper = np.full((sequence_len, sequence_len), -np.inf)
        self.buffers["mask"] = Tensor(np.triu(upper, k=1).astype(self.dtype))

        freqs = config["rope_theta"] ** (
            -2 * np.arange(self.head_dim // 2) / self.head_dim
        )
        theta = np.outer(np.arange(sequence_len), freqs)
        cos, sin = np.cos(theta), np.sin(theta)

        self.buffers["rope1"] = Tensor(
            np.stack((cos, cos), axis=-1)
            .reshape(sequence_len, self.head_dim)
            .astype(self.dtype)
        )
        self.buffers["rope2"] = Tensor(
            np.stack((-sin, sin), axis=-1)
            .reshape(sequence_len, self.head_dim)
            .astype(self.dtype)
        )
        ### END YOUR CODE


    @mugrade.submit_tests
    def __call__(self, tokens, pos=0, cache=None):
        ### BEGIN YOUR CODE
        _, L = tokens.shape
        x = rms_norm(embedding(tokens, self.params["embedding"], self.dtype))
        mask = self.buffers["mask"][pos:pos + L, :pos + L]

        for layer in range(self.num_layers):
            x = self.transformer_block(
                x, layer, mask, pos=pos, cache=cache
            )

        return linear(rms_norm(x), self.params["output"])
        ### END YOUR CODE

Mugrade: Submitting tests for function embedding():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function linear():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function rope():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function multihead_attn():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function mlp():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function transformer_block():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function LLM_init():
Grader test 1 passed
Grader test 2 passed
Grader test 3 failed
Grader test 4 passed
Mugrade: Submitting tests for function __call__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed


### Validate gradients

Finally, the following code will compute the same gradients as we did above with PyTorch, but using your own library.  If you did everything correctly, then these will be the same as the PyTorch gradients up to floating point precision.

In [ ]:
import json
import torch  # Used only to load the saved PyTorch tensors.
from array import array

with open("config.d12.json", "rt") as f: config = json.load(f)
# CuPy has no bfloat16 dtype, so the CuPy model runs in float32. The PyTorch
# reference ran in the config dtype, so expect small numerical differences.
config["dtype"] = "float32"
llm = LLM(config)
params = torch.load("llm.d12.pt", map_location="cpu")
for k,v in params.items():
    llm.params[k] = Tensor(np.array(v.detach().float().numpy()), requires_grad=True)

## compute gradients for parameters on the first 4 sequences in fineweb
with open("fineweb-edu-10BT.shuffle.bin", "rb") as f:
    tokens = array("H", f.read(4*(2048+1)*2)).tolist()
tokens = np.array(tokens).reshape(4,2049)

out = llm(tokens[:,:-1])
loss = cross_entropy_loss(out, tokens[:,1:])
loss.backward()

true_grads = torch.load("grads.pt", map_location="cpu")
for k,p in llm.params.items():
    print(k, np.linalg.norm(p.grad - np.array(true_grads[k].float().numpy())))